# ДЗ-11

## 1. постановка задачи

матрица $A$ размера $8 \times 8$ строится из sha-256 хэша строки Бабаева А.В. 501290. шестьдесят четыре шестнадцатеричные цифры хэша записываются по строкам слева направо сверху вниз и переводятся в десятичную систему.

для матрицы $M = A A^{T}$ требуется реализовать метод вращений якоби и найти все собственные числа и собственные векторы 

матрица $M = A A^{T}$ симметрична, поэтому к ней применим метод вращений якоби, основанный на приведении матрицы к диагональному виду цепочкой преобразований подобия с матрицами плоских вращений.

## 2. построение матрицы

In [ ]:
import numpy as np
import hashlib

In [ ]:
s = 'Бабаева А.В. 501290'
h = hashlib.sha256(s.encode('utf-8')).hexdigest()
print('sha-256:', h)

A = np.array([int(c, 16) for c in h]).reshape(8, 8).astype(float)
print('матрица A:')
print(A.astype(int))

In [ ]:
M = A @ A.T
print('матрица M = A A^T:')
print(M.astype(int))
print()
print('симметричность M:', np.allclose(M, M.T))

## 3. метод вращений якоби

на каждом шаге выбирается ключевой элемент: наибольший по модулю внедиагональный элемент текущей матрицы. через него определяется угол вращения, обнуляющий этот элемент.

пусть ключевой элемент стоит на позиции $(i, j)$. вычисляются величины

$$p = 2 a_{ij}, \qquad q = a_{ii} - a_{jj}, \qquad d = \sqrt{p^{2} + q^{2}}.$$

параметры вращения $c = \cos\alpha$ и $s = \sin\alpha$ находятся по формулам тригонометрии без явного вычисления угла:

$$r = \frac{|q|}{2d}, \qquad c = \sqrt{0.5 + r}, \qquad s = \sqrt{0.5 - r} \cdot \mathrm{sgn}(p q).$$

матрица плоского вращения $T$ совпадает с единичной всюду, кроме элементов $T_{ii} = T_{jj} = c$, $T_{ij} = -s$, $T_{ji} = s$. новая матрица получается преобразованием подобия $B = T^{T} M T$, при этом элемент на позиции $(i, j)$ обнуляется.

признаком окончания служит малость нормы внедиагональной части матрицы. собственными векторами являются столбцы накопленного произведения матриц вращений.

In [ ]:
def off_norm(B):
    return np.sqrt(np.sum(B**2) - np.sum(np.diag(B)**2))

In [ ]:
n = M.shape[0]
eps = 1e-9

B = M.copy()
V = np.eye(n)

for step in range(1000):
    i, j, mx = 0, 1, 0.0
    for p in range(n):
        for q in range(p+1, n):
            if abs(B[p, q]) > mx:
                mx = abs(B[p, q])
                i, j = p, q
    if off_norm(B) < eps:
        break
    p_ = 2*B[i, j]
    q_ = B[i, i] - B[j, j]
    d = np.sqrt(p_**2 + q_**2)
    if abs(q_) < 1e-300:
        c = s = np.sqrt(2)/2
    else:
        r = abs(q_) / (2*d)
        c = np.sqrt(0.5 + r)
        s = np.sqrt(0.5 - r) * np.sign(p_ * q_)
    T = np.eye(n)
    T[i, i] = c
    T[j, j] = c
    T[i, j] = -s
    T[j, i] = s
    B = T.T @ B @ T
    V = V @ T

print('число шагов:', step+1)
print('норма внедиагональной части итоговой матрицы:', f'{off_norm(B):.2e}')

## 4. результат и проверка

собственные числа стоят на диагонали итоговой матрицы. выпишем их в порядке возрастания.

In [ ]:
eigvals = np.diag(B)
poryadok = np.argsort(eigvals)
eigvals_sorted = eigvals[poryadok]
V_sorted = V[:, poryadok]

print('собственные числа (по возрастанию):')
for val in eigvals_sorted:
    print(f'  {val:.4f}')

In [ ]:
print('собственные векторы (столбцы, в том же порядке):')
print(np.round(V_sorted, 4))

проверка ортогональности собственных векторов и диагонализации: матрица $V^{T} M V$ должна быть диагональной. дополнительно собственные числа сравниваются с контрольным расчётом.

In [ ]:
D = V_sorted.T @ M @ V_sorted
print('норма внедиагональной части V^T M V:', f'{off_norm(D):.2e}')
print()
kontrol = np.linalg.eigvalsh(M)
print('собственные числа по контрольному расчёту:')
print(np.round(kontrol, 4))
print()
print('максимальное расхождение:', f'{np.max(np.abs(eigvals_sorted - kontrol)):.2e}')

## 5. выводы

метод вращений якоби привёл матрицу $M = A A^{T}$ к диагональному виду и дал все восемь собственных чисел. норма внедиагональной части итоговой матрицы имеет порядок машинного нуля, а матрица $V^{T} M V$ диагональна, что подтверждает ортогональность набора собственных векторов и корректность диагонализации. найденные собственные числа совпадают с контрольным расчётом с точностью до машинной погрешности.

наибольшее собственное число приблизительно равно $4498.70$ и совпадает с результатом степенного метода из задания №10. при выборе ключевого элемента по максимальному модулю последовательность матриц сходится к диагональной со скоростью геометрической прогрессии.